# GIAI ĐOẠN 1: THU THẬP & KIỂM ĐỊNH DỮ LIỆU BCTC CHUẨN VAS VIỆT NAM

**Dự án:** `FinAudit AI — FinForensics` (SE-Hackathon Challenge 2026 — Trường Kinh tế, ĐH Công nghiệp Hà Nội)  
**Mục tiêu Notebook:** Giúp người mới hiểu rõ từng bước lấy dữ liệu Báo cáo Tài chính thật của toàn bộ doanh nghiệp **phi tài chính** niêm yết trên HOSE & HNX (giai đoạn 2018–2025), tự kiểm chứng phương trình kế toán và phân tích trực quan cấu trúc dữ liệu trước khi sang Giai đoạn 2.

> **Lưu ý pháp lý:** Dữ liệu lịch sử 2018–2025 được doanh nghiệp lập theo **Thông tư 200/2014/TT-BTC** và đã được chuẩn hóa theo mã số chỉ tiêu cố định, liên thông trực tiếp với **Thông tư 99/2025/TT-BTC** (hiệu lực từ `01/01/2026`, đổi tên *Bảng CĐKT* thành *Báo cáo Tình hình Tài chính — Mẫu B01-DN*).

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'stage_1_data_acquisition' else Path.cwd()
RAW_DIR = ROOT_DIR / 'data_store' / 'raw'
STAGE_DIR = ROOT_DIR / 'stage_1_data_acquisition'

print('Thư mục gốc dự án:', ROOT_DIR)
print('Thư mục dữ liệu thô:', RAW_DIR)

## Bước 1: Kiểm tra Phễu Lọc Mẫu Toàn Thị trường (Sample Selection Funnel)
Nếu chưa chạy thu thập dữ liệu, bạn chạy lệnh `python stage_1_data_acquisition/01_data_acquisition.py` trước. Ở bước này ta đọc file nhật ký thu thập và danh mục cổ phiếu để kiểm chứng quy trình lọc bỏ UPCoM và các Định chế Tài chính (Ngân hàng, Chứng khoán, Bảo hiểm).

In [ ]:
with open(RAW_DIR / 'stage1_acquisition_log.json', 'r', encoding='utf-8') as f:
    acq_log = json.load(f)

funnel = acq_log['funnel']
funnel_df = pd.DataFrame([
    ('1. Toàn thị trường (HOSE + HNX + UPCoM)', funnel['total_market_all_exchanges']),
    ('2. Lọc 2 sàn chính thức (HOSE & HNX)', funnel['hose_hnx_total']),
    ('3. Loại Định chế Tài chính (Ngân hàng/CK/BH)', -funnel['excluded_financial_institutions']),
    ('4. Tập mục tiêu Phi tài chính (HOSE & HNX)', funnel['target_non_financial_hose_hnx']),
    ('5. Thu thập thành công đủ 3 bảng BCTC', acq_log['n_companies_success']),
], columns=['Bước lọc khoa học', 'Số doanh nghiệp'])
display(funnel_df)
print(f"Tổng số quan sát firm-year (2018-2025): {acq_log['n_firm_years']:,} dòng | Thời gian tải: {acq_log['elapsed_seconds']}s")

## Bước 2: Khám phá Cấu trúc Bảng BCTC 35 Chỉ tiêu Chuẩn hóa (Đơn vị: Tỷ VNĐ)
Đọc file Parquet siêu nhẹ và xem trực tiếp dữ liệu thực tế của 3 doanh nghiệp đầu ngành (`FPT`, `VNM`, `HPG`) cùng 3 doanh nghiệp từng có biến động dòng tiền/công nợ lớn (`NVL`, `HBC`, `HAG`).

In [ ]:
df = pd.read_parquet(RAW_DIR / 'vas_financial_statements_raw.parquet')
print(f'Kích thước bảng dữ liệu: {df.shape[0]:,} dòng x {df.shape[1]} cột')

spotlight = ['FPT', 'VNM', 'HPG', 'NVL', 'HBC', 'HAG']
cols_view = [
    'ticker', 'exchange', 'icb_sector_vi', 'year', 'total_assets', 'total_liabilities',
    'total_equity', 'net_revenue', 'gross_profit', 'net_income', 'cfo', 'provision_doubtful_debts'
]
display(df[(df['ticker'].isin(spotlight)) & (df['year'] == 2025)][cols_view].reset_index(drop=True))

## Bước 3: Tự Kiểm chứng Phương trình Kế toán (Quality Gate 1)
Kiểm tra 2 đẳng thức sống còn trên **100% tập dữ liệu**:
1. `B01-DN`: $\text{Tổng Tài sản} = \text{Nợ phải trả} + \text{Vốn chủ sở hữu}$
2. `B02-DN`: $\text{Lợi nhuận gộp} = \text{Doanh thu thuần} - \text{Giá vốn hàng bán}$

In [ ]:
valid_bs = df['total_assets'].notna() & (df['total_assets'] > 0) & df['total_liabilities'].notna() & df['total_equity'].notna()
bs_err = (df.loc[valid_bs, 'total_assets'] - (df.loc[valid_bs, 'total_liabilities'] + df.loc[valid_bs, 'total_equity'])).abs() / df.loc[valid_bs, 'total_assets'] * 100

valid_is = df['net_revenue'].notna() & (df['net_revenue'].abs() > 0.001) & df['cogs'].notna() & df['gross_profit'].notna()
gp_err = (df.loc[valid_is, 'gross_profit'] - (df.loc[valid_is, 'net_revenue'] - df.loc[valid_is, 'cogs'])).abs() / df.loc[valid_is, 'net_revenue'].abs() * 100

print(f'1. Tỷ lệ khớp B01-DN (sai số < 1%): {(bs_err < 1.0).mean()*100:.2f}% ({(bs_err < 1.0).sum():,}/{valid_bs.sum():,} dòng)')
print(f'2. Tỷ lệ khớp B02-DN (sai số < 1%): {(gp_err < 1.0).mean()*100:.2f}% ({(gp_err < 1.0).sum():,}/{valid_is.sum():,} dòng)')
print(f'3. Số quan sát có Phân kỳ dòng tiền (LNST > 0 nhưng CFO < 0): {((df["net_income"] > 0) & (df["cfo"] < 0)).sum():,} dòng ({((df["net_income"] > 0) & (df["cfo"] < 0)).mean()*100:.2f}%)')
print(f'4. Số quan sát có Vốn chủ sở hữu âm (giữ nguyên để phát hiện kiệt quệ): {(df["total_equity"] < 0).sum():,} dòng')

## Bước 4: Trực quan hóa Phân bố Dữ liệu theo 16 Nhóm Ngành ICB & Chuỗi Thời gian
Xuất biểu đồ trực quan lưu tại `stage_1_data_acquisition/stage1_data_overview.png` để phục vụ đối chứng và đưa vào báo cáo dự thi.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15.5, 6.2), dpi=140)
fig.patch.set_facecolor('#F8FAFC')

# Biểu đồ 1: Số doanh nghiệp theo ngành ICB
sec_counts = df.groupby('icb_sector_vi')['ticker'].nunique().sort_values(ascending=True)
colors_sec = ['#DC2626' if v < 10 else '#1E40AF' for v in sec_counts.values]
axes[0].barh(sec_counts.index, sec_counts.values, color=colors_sec, edgecolor='none', alpha=0.92)
axes[0].set_xlim(0, sec_counts.max() * 1.14)
axes[0].set_title(f'Phân bố {df["ticker"].nunique()} DN Phi tài chính theo 16 Ngành ICB (HOSE & HNX)', fontsize=11, fontweight='bold', color='#0F172A', pad=12)
axes[0].set_xlabel('Số lượng doanh nghiệp', fontsize=9.5)
axes[0].axvline(10, color='#D97706', linestyle='--', linewidth=1.3, label='Ngưỡng mẫu tối thiểu ngành (n = 10)')
axes[0].legend(fontsize=8.5, loc='lower right', frameon=True, facecolor='white')
axes[0].grid(axis='x', linestyle=':', alpha=0.5)
axes[0].set_facecolor('#FFFFFF')
for i, v in enumerate(sec_counts.values):
    axes[0].text(v + 1.8, i, str(v), va='center', fontsize=8.5, color='#0F172A', fontweight='bold')

# Biểu đồ 2: Số doanh nghiệp & tỷ lệ phân kỳ dòng tiền theo năm
yr_df = df.groupby('year').agg(
    n_co=('ticker', 'nunique'),
    n_div=('cfo', lambda s: int(((df.loc[s.index, 'net_income'] > 0) & (s < 0)).sum()))
).reset_index()
x = np.arange(len(yr_df))
axes[1].bar(x - 0.19, yr_df['n_co'], width=0.38, color='#1E40AF', label='Tổng số DN có đủ 3 bảng BCTC')
axes[1].bar(x + 0.19, yr_df['n_div'], width=0.38, color='#DC2626', label='DN báo Lãi dương (NI > 0) nhưng CFO âm')
axes[1].set_ylim(0, yr_df['n_co'].max() * 1.22)
axes[1].set_xticks(x)
axes[1].set_xticklabels(yr_df['year'].astype(str))
axes[1].set_title('Số lượng DN & Hiện tượng Phân kỳ Dòng tiền (2018–2025)', fontsize=11, fontweight='bold', color='#0F172A', pad=12)
axes[1].set_ylabel('Số lượng doanh nghiệp', fontsize=9.5)
axes[1].legend(fontsize=8.5, loc='upper center', ncol=2, frameon=True, facecolor='white')
axes[1].grid(axis='y', linestyle=':', alpha=0.5)
axes[1].set_facecolor('#FFFFFF')
for i, (tot, div) in enumerate(zip(yr_df['n_co'], yr_df['n_div'])):
    axes[1].text(i - 0.19, tot + 10, str(tot), ha='center', fontsize=8.2, fontweight='bold')
    axes[1].text(i + 0.19, div + 10, str(div), ha='center', fontsize=8.2, color='#DC2626', fontweight='bold')

plt.tight_layout()
chart_path = STAGE_DIR / 'stage1_data_overview.png'
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print('Đã lưu biểu đồ tổng quan tại:', chart_path)